In [1]:
# Importar librerias
import pandas as pd

from funciones import *

In [2]:
# Generar lista de tickers del Fichero constituents del S&P 500
data_folder = "../data"
df_tickers = pd.read_csv(f"{data_folder}/constituents.csv")

# Modificar "BRK.B" a "BRK-B" y "BF.B" a "BF-B" para evitar problemas con yfinance
df_tickers["Symbol"] = df_tickers["Symbol"].replace("BRK.B", "BRK-B")
df_tickers["Symbol"] = df_tickers["Symbol"].replace("BF.B", "BF-B")

tickers_list = df_tickers["Symbol"].tolist()
len(tickers_list)

503

In [7]:
# Seleccionar y renombrar columnas en df_tickers
df_tickers = df_tickers[["Symbol", "GICS Sub-Industry", "Date added"]]
df_tickers.rename(columns={
    "Symbol": "Ticker",
    "GICS Sub-Industry": "SubIndustry", 
    "Date added": "DateAdded"
    }, inplace=True)

In [8]:
df_tickers.head()

,Ticker,SubIndustry,DateAdded
0,MMM,Industrial Conglomerates,1957-03-04
1,AOS,Building Products,2017-07-26
2,ABT,Health Care Equipment,1957-03-04
3,ABBV,Biotechnology,2012-12-31
4,ACN,IT Consulting & Other Services,2011-07-06


In [9]:
df_tickers.SubIndustry.value_counts()

SubIndustry
Health Care Equipment                           17
Electric Utilities                              15
Application Software                            14
Semiconductors                                  14
Industrial Machinery & Supplies & Components    14
                                                ..
Health Care Technology                           1
Wireless Telecommunication Services              1
Passenger Ground Transportation                  1
Timber REITs                                     1
Homefurnishing Retail                            1
Name: count, Length: 127, dtype: int64

In [3]:
# Extraer info desde la API de yfinance (demora unos 6 minutos)
df_info = extraer_info(tickers_list)
df_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Ticker               503 non-null    object 
 1   Sector               502 non-null    object 
 2   MarketCap            503 non-null    int64  
 3   Beta                 497 non-null    float64
 4   DividendYield        409 non-null    float64
 5   ForwardPE            503 non-null    float64
 6   trailingPegRatio     435 non-null    float64
 7   PriceToBook          503 non-null    float64
 8   EnterpriseToEbitda   472 non-null    float64
 9   ReturnOnAssets       499 non-null    float64
 10  returnOnEquity       470 non-null    float64
 11  profitMargins        502 non-null    float64
 12  operatingMargins     502 non-null    float64
 13  currentRatio         482 non-null    float64
 14  debtToEquity         452 non-null    float64
 15  revenueGrowth        502 non-null    flo

In [10]:
# Unir df_info con df_tickers
df_merged = pd.merge(df_info, df_tickers, on="Ticker", how="left")

# Guardar datos extraidos en fichero raw_data
df_merged.to_parquet(f"{data_folder}/raw_data.parquet")